[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/21_batch_inference.ipynb)

# Batch inference — run your fine-tuned model over a new dataset

> **Fine-tuning series — 21 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.

## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# A tiny instruction set, only used to fine-tune a throwaway adapter below so the
# notebook has a model to load. Your real artifact comes from notebooks 04/05/08/20.
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos d\u00edas."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
print(sft_ds[0]["text"][:160])

In [ ]:
# Default LoRA config (same one reused across the series).
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## Why a separate inference notebook

Saving a fine-tune (notebook 17) and *evaluating* it on a labeled holdout (notebook 19)
are not the same as the everyday job: **you have a saved model and a pile of new, unlabeled
inputs, and you want its predictions** — fast, batched, and written to a file you can use.

Three things make batch inference different from a single `generate()` call:

- **Padding side must be `left`** for decoder-only models, or batched generations are garbage.
- **Batch size trades speed for memory** — bigger batches are faster until they OOM.
- **Fix decoding** (`do_sample=False`) so a re-run gives the same predictions.

### 0. Make a saved fine-tune to load

Quick throwaway adapter so the notebook is self-contained. **In your own work, skip this** and point `SAVED_DIR` at the adapter you trained in notebook 04/05/08/20.

In [ ]:
from trl import SFTTrainer, SFTConfig
m = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
_tr = SFTTrainer(model=m, train_dataset=sft_ds, peft_config=lora_cfg,
    args=SFTConfig(output_dir="bi_train", dataset_text_field="text",
                   per_device_train_batch_size=2, max_steps=20, learning_rate=2e-4,
                   logging_steps=10, bf16=BF16_OK, report_to="none"))
_tr.train()

SAVED_DIR = "my_finetuned_adapter"          # <- your real adapter path goes here
_tr.save_model(SAVED_DIR)
tokenizer.save_pretrained(SAVED_DIR)
del m, _tr; cleanup()
print("saved fine-tuned adapter ->", SAVED_DIR)

### 1. Load the saved fine-tune for inference

Base model + adapter, then `merge_and_unload()` to fold the adapter in — one plain model, no per-call adapter overhead. (To keep adapters hot-swappable instead, skip the merge and keep the `PeftModel`.)

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device),
    SAVED_DIR)
model = model.merge_and_unload().eval()      # adapter folded into the base weights

tokenizer.padding_side = "left"              # REQUIRED for batched decoder-only generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("loaded + merged; padding side:", tokenizer.padding_side)

### 2. A batched `generate()` helper

Apply the chat template to each prompt, tokenize the chunk with padding, generate once for the whole batch, then slice off the prompt tokens. With left padding every prompt ends at the same column, so one slice index works for the batch.

In [ ]:
@torch.no_grad()
def generate_batch(prompts, max_new_tokens=64, batch_size=8):
    out = []
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        texts = [tokenizer.apply_chat_template(
                    [{"role": "user", "content": p}],
                    tokenize=False, add_generation_prompt=True) for p in chunk]
        enc = tokenizer(texts, return_tensors="pt", padding=True,
                        truncation=True, max_length=1024).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]            # drop the (padded) prompt
        out += [t.strip() for t in tokenizer.batch_decode(new, skip_special_tokens=True)]
    return out

print(generate_batch(["Name a primary color.", "What is 2 + 2?"], max_new_tokens=16))

### 3a. New data from the Hub

The common case: a Hugging Face dataset of inputs you want answered. Here we take a slice of a real instruction set but use **only the questions** — pretend the answers don't exist.

In [ ]:
from datasets import load_dataset

new = load_dataset("tatsu-lab/alpaca", split="train[:12]")   # a fresh batch of inputs
prompts = [r["instruction"] + (("\n" + r["input"]) if r["input"].strip() else "")
           for r in new]

preds = generate_batch(prompts, max_new_tokens=48, batch_size=4)
for p, a in list(zip(prompts, preds))[:4]:
    print("Q:", p[:70].replace("\n", " "))
    print("  ->", a[:90], "\n")

# Attach predictions back onto the dataset and (optionally) push/save it.
scored = new.add_column("prediction", preds)
print("columns now:", scored.column_names)

### 3b. New data from your own CSV / JSONL

Load → infer → **save predictions back to a file**. JSONL shown in full; the CSV equivalents are one-liners in the comments.

In [ ]:
import json, csv

# Write a tiny example input file so the cell runs — replace with your real path.
rows = [{"id": 1, "prompt": "Summarize in one line: the cat sat on the mat."},
        {"id": 2, "prompt": "Translate 'thank you' to French."},
        {"id": 3, "prompt": "Name the largest ocean."}]
with open("new_inputs.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")

# --- load (JSONL; CSV: list(csv.DictReader(open("new_inputs.csv")))) ---
data = [json.loads(line) for line in open("new_inputs.jsonl")]

# --- infer ---
answers = generate_batch([d["prompt"] for d in data], max_new_tokens=48, batch_size=4)
for d, a in zip(data, answers):
    d["prediction"] = a

# --- save (JSONL + CSV; with pandas: pd.DataFrame(data).to_csv("predictions.csv")) ---
with open("predictions.jsonl", "w") as f:
    for d in data:
        f.write(json.dumps(d) + "\n")
with open("predictions.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(data[0].keys()))
    w.writeheader(); w.writerows(data)

print("wrote predictions.jsonl + predictions.csv")
print(data[0])

### Recap & pitfalls

- **`padding_side = "left"`** before batching — the single most common cause of garbled batched output.
- **Tune `batch_size`** to your VRAM: raise it for throughput, lower it if you OOM (notebook 18).
- **`do_sample=False`** (greedy) for reproducible predictions; set a seed if you sample.
- **`truncation=True, max_length=...`** so one giant input can't blow up the batch.
- **Merge once, infer many:** `merge_and_unload()` is worth it for a big run; keep the `PeftModel` if you hot-swap adapters.
- For high-throughput serving (continuous batching, paged attention) reach for **vLLM / TGI** — see notebook 17.

In [ ]:
del model; cleanup()